# HoloKai GLiNER — Final Run (mapped data only)

Reuses the Run 1 workflow with `final_model/configs/finetune_final_run.json`: 8 epochs, validation every 100 optimizer steps at threshold 0.40, exact inclusive token-span/label micro P/R/F1. All Final Run artifacts stay on Google Drive under `final_model/outputs`; previous experiments are preserved.

Training, optional analyses, and held-out V4 evaluation are disabled by default. Checkpoints retain optimizer, scheduler, RNG and trainer state. The highest validation-F1 checkpoint (earliest on ties) is exported as `training/best_model`; a final partial interval is also evaluated. Only the best and latest checkpoints are retained.


## Step 1 — Environment setup

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd "/content/drive/MyDrive/Holokai_Project"


In [ ]:
!pip install -r finetune/requirements-colab.txt


## Step 2 — Load and validate existing mapped data

Reuse the existing mapped files without regenerating them. Original files are used only to verify the unchanged label mapping.


In [ ]:
# Mapping was already prepared. Do not regenerate or overwrite datasets.
from pathlib import Path
for name in ("train_gliner_mapping.json", "val_gliner_mapping.json"):
    assert (Path("/content/drive/MyDrive/Holokai_Project/finetune/data") / name).is_file(), name


In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Holokai_Project")

ORIG_TRAIN = PROJECT / "finetune/data/train_gliner.json"
ORIG_VAL   = PROJECT / "finetune/data/val_gliner.json"

TRAIN_DATA = PROJECT / "finetune/data/train_gliner_mapping.json"
VAL_DATA   = PROJECT / "finetune/data/val_gliner_mapping.json"

RUN_ROOT = PROJECT / "final_model/outputs"
TRAIN_OUT = RUN_ROOT / "training"
BASELINE_OUT = RUN_ROOT / "baseline"
MODEL_EVAL_OUT = RUN_ROOT / "model_eval"
COMPARISON_OUT = RUN_ROOT / "comparison"
THRESHOLD_OUT = RUN_ROOT / "threshold_tuning"

LABEL_MAPPING = {
    "provider": "name medical professional",
    "facility": "organization medical facility",
    "person": "name",
    "location address": "location address",
    "diagnosis": "condition",
    "unique identifier": "personal identifier number",
    "sex": "sex assigned at birth",
    "gender identity": "gender identity",
    "procedure": "medical procedure or test performed",
    "medication": "medication, drug, or prescription",
    "lab result": "lab result or test finding",
}

MAPPED_LABELS = list(LABEL_MAPPING.values())

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def validate_mapping(original_path, mapped_path):
    original = load_json(original_path)
    mapped = load_json(mapped_path)

    assert len(original) == len(mapped)

    for i, (a, b) in enumerate(zip(original, mapped)):
        assert a["tokenized_text"] == b["tokenized_text"], f"Tokens changed at sample {i}"
        assert len(a["ner"]) == len(b["ner"]), f"Entity count changed at sample {i}"

        for old, new in zip(a["ner"], b["ner"]):
            assert old[:2] == new[:2], f"Span changed at sample {i}"
            assert old[2] in LABEL_MAPPING, f"Unexpected original label: {old[2]}"
            assert new[2] == LABEL_MAPPING[old[2]], f"Wrong mapping: {old} -> {new}"

    labels = {e[2] for r in mapped for e in r["ner"]}
    assert labels == set(MAPPED_LABELS), f"Unexpected mapped labels: {labels}"

    print(
        f"PASS: {mapped_path.name} | "
        f"samples={len(mapped)} | "
        f"entities={sum(len(r['ner']) for r in mapped)}"
    )

validate_mapping(ORIG_TRAIN, TRAIN_DATA)
validate_mapping(ORIG_VAL, VAL_DATA)

print("\nFormal train data:", TRAIN_DATA)
print("Formal val data:  ", VAL_DATA)


## Step 3 — Formal training config

The mapped train/validation files are the only data paths used from this point forward.

Edit the execution switches in the next cell when ready; rerunning that cell resets them to the values written there. Configuration checking does not start training. Baseline, comparison and threshold tuning are optional.


In [ ]:
import sys
import subprocess
sys.path.insert(0, str(PROJECT / "finetune/scripts"))
TRAIN_SCRIPT = PROJECT / "finetune/scripts/train_gliner_final_run.py"
assert TRAIN_SCRIPT.is_file(), f"Training script missing: {TRAIN_SCRIPT}"
from train_gliner_final_run import load_config

RUN_CONFIG = PROJECT / "final_model/configs/finetune_final_run.json"
cfg = load_config(RUN_CONFIG)
assert cfg["train_data"] == str(TRAIN_DATA)
assert cfg["val_data"] == str(VAL_DATA)
assert cfg["output_dir"] == str(TRAIN_OUT)
assert cfg["epochs"] == 8 and cfg["eval_steps"] == 100
assert cfg["threshold"] == 0.40 and cfg["save_best_model"]
assert cfg["label_schema"] == "recommended"
MAPPED_SCHEMA = "recommended"
RUN_TRAINING = True
RUN_OPTIONAL_ANALYSES = True  # baseline, comparison, validation threshold tuning
RUN_SAVED_VALIDATION = True  # independent evaluation of the exported best model
RESUME_FROM_CHECKPOINT = None  # set to "latest" after an interrupted run
print(json.dumps(cfg, indent=2))

print("Execution switches:", {"RUN_TRAINING": RUN_TRAINING, "RUN_OPTIONAL_ANALYSES": RUN_OPTIONAL_ANALYSES, "RUN_SAVED_VALIDATION": RUN_SAVED_VALIDATION, "RESUME_FROM_CHECKPOINT": RESUME_FROM_CHECKPOINT})


In [ ]:
command = [sys.executable, str(TRAIN_SCRIPT),
           "--config", str(RUN_CONFIG), "--check-config"]
if RESUME_FROM_CHECKPOINT:
    command += ["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT]
subprocess.run(command, check=True)
print("Configuration/data check passed. No training was started; use Step 5 for training.")


## Step 4 — Pretrained baseline on mapped validation data

This is the formal baseline for the fine-tuning experiment.

The baseline evaluator runs in a separate Python process. Its model is not a notebook variable. Read the saved metrics below to inspect the evaluation device and results without rerunning evaluation. The optional GPU diagnostic shows only current usage; memory is normally released after evaluation exits.


In [ ]:
if RUN_OPTIONAL_ANALYSES:
    !python finetune/scripts/evaluate_gliner.py \
      --label-schema recommended \
      --model knowledgator/gliner-pii-base-v1.0 \
      --data "{VAL_DATA}" \
      --threshold 0.4 \
      --output-dir "{BASELINE_OUT}" \
      --device auto
else:
    print("Baseline skipped: RUN_OPTIONAL_ANALYSES=False. Enable it in Step 3 only if needed.")


In [ ]:
# Read the completed baseline result; no model loading or evaluation is performed.
import json

metrics_path = BASELINE_OUT / "metrics.json"
if metrics_path.is_file():
    with metrics_path.open(encoding="utf-8") as handle:
        baseline_metrics = json.load(handle)
    metadata = baseline_metrics.get("metadata", {})
    print("Evaluation device:", metadata.get("device", "Not recorded"))
    print("Sample count:", metadata.get("sample_count", "Not recorded"))
    print("Threshold:", metadata.get("threshold", "Not recorded"))
    print(json.dumps(baseline_metrics["overall"], indent=2))
else:
    print(f"Baseline metrics missing: {metrics_path}.")
    print("The evaluation result has not been saved here. Check the preceding evaluation cell for errors before rerunning it.")


In [ ]:
baseline_files = [BASELINE_OUT / name for name in ("metrics.json", "per_label_metrics.csv")]
if RUN_OPTIONAL_ANALYSES and all(path.is_file() for path in baseline_files):
    import pandas as pd
    
    with open(BASELINE_OUT / "metrics.json", encoding="utf-8") as f:
        baseline_metrics = json.load(f)
    
    print("Pretrained baseline @ threshold=0.4")
    print(json.dumps(baseline_metrics["overall"], indent=2))
    display(pd.read_csv(BASELINE_OUT / "per_label_metrics.csv"))
elif not RUN_OPTIONAL_ANALYSES:
    print("Baseline display skipped: RUN_OPTIONAL_ANALYSES=False.")
else:
    print("Baseline results missing; run the baseline cell first:", [str(path) for path in baseline_files if not path.is_file()])


## Step 5 — Full fine-tuning or resume (explicit opt-in)

Set `RUN_TRAINING=True` in Step 3 when ready. For an interrupted run, also set `RESUME_FROM_CHECKPOINT="latest"`. Resume restores the full state and continues toward the same total of 8 epochs; it does not add 8 more epochs. Config and dataset hashes must match. An existing run cannot be silently restarted. Incomplete checkpoints are ignored. Progress since the last complete checkpoint may be repeated.


In [ ]:
if RUN_TRAINING:
    from datetime import datetime, timezone
    command = [sys.executable, "-u", str(TRAIN_SCRIPT),
               "--config", str(RUN_CONFIG)]
    if RESUME_FROM_CHECKPOINT:
        command += ["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT]
    print("Starting training:", " ".join(command), flush=True)
    log_dir = RUN_ROOT / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (datetime.now(timezone.utc).strftime("train-%Y%m%dT%H%M%S%fZ") + ".log")
    with log_path.open("x", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
else:
    print("Training skipped: RUN_TRAINING=False. Set it to True in Step 3, rerun the config check, then run this cell when ready.")


In [ ]:
if TRAIN_OUT.exists():
    print("Persistent artifacts:", sorted(p.name for p in TRAIN_OUT.iterdir()))
else:
    print(f"Training output directory missing: {TRAIN_OUT}. No training artifacts are available here yet.")


In [ ]:
if (TRAIN_OUT / "training_history.json").exists():
    history = load_json(TRAIN_OUT / "training_history.json")
    
    provider_label = LABEL_MAPPING["provider"]
    
    for row in history:
        if "eval_f1" not in row:
            continue
    
        provider_recall = None
        per_label = row.get("eval_exact_metrics", {}).get("per_label", {})
        if provider_label in per_label:
            provider_recall = per_label[provider_label].get("recall")
    
        print(
            f"step={row.get('step')} | "
            f"epoch={row.get('epoch')} | "
            f"F1={row['eval_f1']:.4f} | "
            f"P={row['eval_precision']:.4f} | "
            f"R={row['eval_recall']:.4f} | "
            f"PROVIDER_R={provider_recall}"
        )
else:
    print(f"Training history missing: {TRAIN_OUT / 'training_history.json'}. Start or resume training explicitly to generate it.")


## Step 6 — Inspect the best export and optionally evaluate saved validation

Selection occurs during training at every 100 steps and at the final step. `best_model.json` records the selected checkpoint and F1. The exported directory contains GLiNER weights, configuration, tokenizer and selection metadata. No epoch re-ranking is needed.


In [ ]:
BEST_MODEL_NAME = "best_model"
BEST_MODEL_PATH = TRAIN_OUT / BEST_MODEL_NAME
if (TRAIN_OUT / "best_model.json").exists():
    print(json.dumps(load_json(TRAIN_OUT / "best_model.json"), indent=2))
else:
    print(f"Best-model selection record missing: {TRAIN_OUT / 'best_model.json'}. Complete training/export first.")
if RUN_SAVED_VALIDATION and BEST_MODEL_PATH.is_dir():
    subprocess.run([sys.executable, str(PROJECT / "finetune/scripts/evaluate_gliner.py"),
                    "--label-schema", "recommended", "--model", str(BEST_MODEL_PATH),
                    "--data", str(VAL_DATA), "--threshold", "0.40", "--batch-size", "8",
                    "--output-dir", str(MODEL_EVAL_OUT / BEST_MODEL_NAME), "--device", "auto"], check=True)
elif not RUN_SAVED_VALIDATION:
    print("Saved-model validation skipped: RUN_SAVED_VALIDATION=False. Training-time validation is independent of this switch.")
else:
    print(f"Saved-model validation unavailable: model directory missing: {BEST_MODEL_PATH}. Complete training/export first.")


In [ ]:
metric_files = [TRAIN_OUT / name for name in ("validation_metrics.json", "train_loss.json")]
if all(path.is_file() for path in metric_files):
    import pandas as pd
    display(pd.DataFrame(load_json(TRAIN_OUT / "validation_metrics.json"))[
        ["step", "epoch", "eval_precision", "eval_recall", "eval_f1"]])
    display(pd.DataFrame(load_json(TRAIN_OUT / "train_loss.json")))
else:
    print("Training metric files missing:", [str(path) for path in metric_files if not path.is_file()])


## Step 7 — Pretrained vs fine-tuned comparison

Both sides use:
- mapped labels,
- mapped validation data,
- threshold `0.4`,
- identical exact-span evaluation.


In [ ]:
comparison_files = [directory / name for directory in (BASELINE_OUT, MODEL_EVAL_OUT / BEST_MODEL_NAME)
                    for name in ("metrics.json", "per_label_metrics.csv")]
if RUN_OPTIONAL_ANALYSES and all(path.is_file() for path in comparison_files):
    import pandas as pd
    COMPARISON_OUT.mkdir(parents=True, exist_ok=True)
    
    with open(BASELINE_OUT / "metrics.json", encoding="utf-8") as f:
        base_overall = json.load(f)["overall"]
    
    with open(MODEL_EVAL_OUT / BEST_MODEL_NAME / "metrics.json", encoding="utf-8") as f:
        ft_overall = json.load(f)["overall"]
    
    overall_comparison = {
        "threshold": 0.4,
        "pretrained": base_overall,
        "fine_tuned": ft_overall,
        "delta": {
            "precision": ft_overall["precision"] - base_overall["precision"],
            "recall": ft_overall["recall"] - base_overall["recall"],
            "f1": ft_overall["f1"] - base_overall["f1"],
        },
    }
    
    with open(COMPARISON_OUT / "pretrained_vs_finetuned_overall.json", "w", encoding="utf-8") as f:
        json.dump(overall_comparison, f, indent=2)
    
    base_df = pd.read_csv(BASELINE_OUT / "per_label_metrics.csv")
    ft_df = pd.read_csv(MODEL_EVAL_OUT / BEST_MODEL_NAME / "per_label_metrics.csv")
    
    comparison_df = base_df.merge(
        ft_df,
        on="label",
        suffixes=("_pretrained", "_finetuned"),
    )
    
    for metric in ["precision", "recall", "f1"]:
        comparison_df[f"{metric}_delta"] = (
            comparison_df[f"{metric}_finetuned"]
            - comparison_df[f"{metric}_pretrained"]
        )
    
    comparison_df.to_csv(
        COMPARISON_OUT / "pretrained_vs_finetuned_per_label.csv",
        index=False
    )
    
    print(json.dumps(overall_comparison, indent=2))
    display(comparison_df)
elif not RUN_OPTIONAL_ANALYSES:
    print("Comparison skipped: RUN_OPTIONAL_ANALYSES=False.")
else:
    print("Comparison inputs missing:", [str(path) for path in comparison_files if not path.is_file()])
    print("Run the baseline and enable RUN_SAVED_VALIDATION to evaluate the exported model first.")


## Step 8 — Per-label threshold tuning

Tune thresholds only for the selected best fine-tuned model, using the mapped validation data.


In [ ]:
if RUN_OPTIONAL_ANALYSES and BEST_MODEL_PATH.exists():
    !python finetune/scripts/tune_per_label_thresholds.py \
      --label-schema "{MAPPED_SCHEMA}" \
      --model "{BEST_MODEL_PATH}" \
      --data "{VAL_DATA}" \
      --output-dir "{THRESHOLD_OUT}"
elif not RUN_OPTIONAL_ANALYSES:
    print("Threshold tuning skipped: RUN_OPTIONAL_ANALYSES=False.")
else:
    print(f"Threshold tuning unavailable: model directory missing: {BEST_MODEL_PATH}. Complete training/export first.")


In [ ]:
threshold_files = [THRESHOLD_OUT / name for name in ("best_thresholds.json", "calibrated_metrics.json", "calibrated_per_label_metrics.csv")]
if RUN_OPTIONAL_ANALYSES and all(path.is_file() for path in threshold_files):
    import pandas as pd
    best_thresholds = load_json(THRESHOLD_OUT / "best_thresholds.json")
    calibrated_metrics = load_json(THRESHOLD_OUT / "calibrated_metrics.json")
    
    print("Best thresholds:")
    print(json.dumps(best_thresholds, indent=2))
    
    print("\nCalibrated validation overall:")
    print(json.dumps(calibrated_metrics["overall"], indent=2))
    
    display(pd.read_csv(THRESHOLD_OUT / "calibrated_per_label_metrics.csv"))
elif not RUN_OPTIONAL_ANALYSES:
    print("Threshold results display skipped: RUN_OPTIONAL_ANALYSES=False.")
else:
    print("Threshold results missing; run threshold tuning first:", [str(path) for path in threshold_files if not path.is_file()])


In [ ]:
# Load overall validation metrics
fixed = load_json(
    MODEL_EVAL_OUT / BEST_MODEL_NAME / "metrics.json"
)["overall"]

tuned = load_json(
    THRESHOLD_OUT / "calibrated_metrics.json"
)["overall"]

# Display comparison
rows = []

for name, metrics in [
    ("Fine-tuned (Fixed 0.40)", fixed),
    ("Fine-tuned (Tuned Thresholds)", tuned)
]:
    rows.append({
        "Evaluation": name,
        "Precision": f"{metrics['precision']:.2%}",
        "Recall": f"{metrics['recall']:.2%}",
        "F1": f"{metrics['f1']:.2%}",
        "TP": metrics["tp"],
        "FP": metrics["fp"],
        "FN": metrics["fn"],
        "Support": metrics["support"]
    })

print("=== OVERALL VALIDATION RESULTS ===")
display(pd.DataFrame(rows))

## Step 9 — Inspect saved run provenance

Training saves the resolved configuration, dataset hashes, loss/metric histories, trainer state, full resumable checkpoints and best-model selection metadata. Console logs are under `final_model/outputs/logs`. Optional baseline/comparison/threshold-tuning results use their existing formats. V4 is never used for model selection or tuning.


In [ ]:
for name in ("training_config.json", "data_manifest.json", "best_model.json"):
    path = TRAIN_OUT / name
    if path.exists():
        print(name)
        print(json.dumps(load_json(path), indent=2))
    else:
        print(f"Run artifact missing: {path}. Training/export may not have completed yet.")


In [ ]:
# Independent reload: in a fresh runtime, run only Drive mount, dependency installation,
# and this cell. It does not depend on training variables or load any datasets.
from pathlib import Path
import torch
from gliner import GLiNER

SAVED_MODEL = Path("/content/drive/MyDrive/Holokai_Project/final_model/outputs/training/best_model")
if SAVED_MODEL.is_dir():
    inference_model = GLiNER.from_pretrained(str(SAVED_MODEL), local_files_only=True)
    inference_model.to("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.eval()
    print("Independently reloaded:", SAVED_MODEL)
else:
    print(f"Model export missing: {SAVED_MODEL}. Complete training/export before independently reloading.")


## Step 10 — Held-out V4 Test Evaluation
Run after freezing the Final Run model and the validation-selected thresholds from Step 8. These cells need only the installed dependencies and mounted Google Drive; Steps 1–9 do not need to be rerun.

The fixed threshold 0.40 evaluation keeps its original `outputs/v4_test` directory and results. The second evaluation reads `outputs/threshold_tuning/best_thresholds.json` unchanged, decodes at its recorded candidate threshold (0.10), and applies the existing post-decoding per-label filter. No V4 threshold search or model selection is performed.

Per-label results go to `outputs/v4_test_per_label`; overall comparisons go to `outputs/v4_test_comparison`. Complete matching results are reused, partial or incompatible results are rejected, and existing files are never overwritten. The comparison reports per-label-threshold results minus fixed-0.40 results; rate differences are absolute, not relative percentages.

In [ ]:
RUN_V4_EVALUATION = True
import sys
import subprocess
import json
import hashlib
from pathlib import Path

project = Path("/content/drive/MyDrive/Holokai_Project")
sys.path.insert(0, str(project / "finetune/scripts"))
saved_model = project / "final_model/outputs/training/best_model"
v4_data = project / "finetune/data/corpus_V4_gliner_mapping.json"
v4_fixed_out = project / "final_model/outputs/v4_test"
v4_per_label_out = project / "final_model/outputs/v4_test_per_label"
threshold_file = project / "final_model/outputs/threshold_tuning/best_thresholds.json"

# Verify the APIs in the Google Drive scripts actually used by this Colab run.
# Older copies lack load_selected_thresholds and --thresholds-file; neither is required below.
if RUN_V4_EVALUATION:
    import importlib
    import pandas as pd
    from IPython.display import display
    import torch
    from gliner import GLiNER
    evaluator_script = project / "finetune/scripts/evaluate_gliner.py"
    tuner_script = project / "finetune/scripts/tune_per_label_thresholds.py"
    for path in (evaluator_script, tuner_script, v4_data, threshold_file):
        if not path.is_file():
            raise FileNotFoundError(f"Step 10 dependency missing: {path}")
    if not saved_model.is_dir():
        raise FileNotFoundError(f"Saved best model missing: {saved_model}")
    for module_name, expected_path in (("evaluate_gliner", evaluator_script),
                                       ("tune_per_label_thresholds", tuner_script)):
        cached = sys.modules.get(module_name)
        if cached is not None and Path(cached.__file__).resolve() != expected_path.resolve():
            sys.modules.pop(module_name)
    evaluator = importlib.import_module("evaluate_gliner")
    tuner = importlib.import_module("tune_per_label_thresholds")
    if Path(evaluator.__file__).resolve() != evaluator_script.resolve():
        raise ImportError(f"Colab loaded a different evaluator: {evaluator.__file__}")
    if Path(tuner.__file__).resolve() != tuner_script.resolve():
        raise ImportError(f"Colab loaded a different threshold script: {tuner.__file__}")
    for name in ("load_data", "predict", "compute_metrics", "save_results"):
        if not callable(getattr(evaluator, name, None)):
            raise ImportError(f"The Colab evaluator is missing {name}: {evaluator_script}")
    if not callable(getattr(tuner, "filter_predictions", None)):
        raise ImportError(f"The Colab threshold script is missing filter_predictions: {tuner_script}")
    import inspect
    for function, parameters in ((evaluator.load_data, ("label_schema",)),
                                 (evaluator.predict, ("flat_ner", "label_schema")),
                                 (evaluator.compute_metrics, ("label_schema",)),
                                 (tuner.filter_predictions, ("label_schema",))):
        missing = set(parameters) - set(inspect.signature(function).parameters)
        if missing:
            raise ImportError(f"The Colab version of {function.__name__} lacks {sorted(missing)}")
    if not hasattr(torch, "inference_mode") or not hasattr(GLiNER, "from_pretrained"):
        raise ImportError("Step 10 requires PyTorch inference_mode and GLiNER.from_pretrained")
    preview = json.loads(threshold_file.read_text())
    if "thresholds" not in preview or "metadata" not in preview:
        raise ValueError(f"Saved validation thresholds are incomplete: {threshold_file}")
    print("Using Colab evaluator:", evaluator_script)
    help_result = subprocess.run([sys.executable, str(evaluator_script), "--help"],
                                 capture_output=True, text=True, check=True)
    for option in ("--label-schema", "--model", "--data", "--threshold", "--output-dir"):
        if option not in help_result.stdout:
            raise RuntimeError(f"The Colab evaluator does not support {option}: {evaluator_script}")

def evaluate_or_load_v4(output, threshold_args, expected):
    required = [output / name for name in ("metrics.json", "per_label_metrics.csv", "predictions.json")]
    if any(path.exists() for path in required):
        if not all(path.is_file() for path in required):
            raise FileExistsError(f"Partial results at {output}; use a fresh directory. Existing files are preserved.")
    else:
        assert saved_model.is_dir(), "Complete and export the Final Run first"
        subprocess.run([sys.executable, str(project / "finetune/scripts/evaluate_gliner.py"),
                        "--label-schema", "recommended", "--model", str(saved_model),
                        "--data", str(v4_data), "--batch-size", "8", "--device", "auto",
                        *threshold_args, "--output-dir", str(output)], check=True)
    result = json.loads(required[0].read_text())
    metadata = result["metadata"]
    checks = {"model": str(saved_model), "data_sha256": hashlib.sha256(v4_data.read_bytes()).hexdigest(),
              "label_schema": "recommended", "flat_ner": False, "multi_label": False, **expected}
    for key, value in checks.items():
        if metadata.get(key) != value:
            raise ValueError(f"Existing result {output} has incompatible {key}; choose a fresh directory")
    print("Results available (existing files preserved):", output)
    return result

if RUN_V4_EVALUATION:
    fixed_v4_metrics = evaluate_or_load_v4(v4_fixed_out, ["--threshold", "0.40"], {"threshold": 0.40})
else:
    print("Held-out V4 evaluation skipped: RUN_V4_EVALUATION=False. Enable only after the model is frozen.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

V4_OUT = Path(
    "/content/drive/MyDrive/Holokai_Project/"
    "final_model/outputs/v4_test"
)

# Load saved evaluation results
with open(V4_OUT / "metrics.json") as f:
    metrics = json.load(f)

per_label = pd.read_csv(
    V4_OUT / "per_label_metrics.csv"
)

# Overall results
print("=== V4 TEST: OVERALL PERFORMANCE ===")

overall = metrics["overall"]

overall_df = pd.DataFrame([{
    "Precision": f"{overall['precision']:.2%}",
    "Recall": f"{overall['recall']:.2%}",
    "F1": f"{overall['f1']:.2%}",
    "TP": overall["tp"],
    "FP": overall["fp"],
    "FN": overall["fn"],
    "Support": overall["support"]
}])

display(overall_df)

# Per-label results
print("\n=== V4 TEST: PER-LABEL PERFORMANCE ===")

label_df = per_label.copy()

for col in ["precision", "recall", "f1"]:
    label_df[col] = label_df[col].map(
        lambda x: f"{x:.2%}"
    )

display(label_df)

In [ ]:
# Use the actual Colab evaluator's existing functions; this also works when its
# copy predates load_selected_thresholds and the --thresholds-file CLI option.
if RUN_V4_EVALUATION:
    from datetime import datetime, timezone
    from importlib.metadata import version
    from label_schemas import get_labels, schema_metadata
    from validate_gliner_data import unique_keys, reject_constant

    raw_thresholds = threshold_file.read_bytes()
    saved_thresholds = json.loads(raw_thresholds.decode("utf-8"),
                                  object_pairs_hook=unique_keys, parse_constant=reject_constant)
    selected = saved_thresholds["thresholds"]
    selection_metadata = saved_thresholds["metadata"]
    threshold_hash = hashlib.sha256(raw_thresholds).hexdigest()
    expected_labels = set(get_labels("recommended"))
    if set(selected) != expected_labels:
        raise ValueError("Saved thresholds must contain exactly the 11 recommended labels")
    if (selection_metadata["label_schema"] != "recommended" or
            selection_metadata["flat_ner"] is not False or
            selection_metadata["multi_label"] is not False or
            Path(selection_metadata["model"]).resolve() != saved_model.resolve() or
            selection_metadata["calibration_method"] !=
            "per-label filtering of a shared post-decoding prediction pool"):
        raise ValueError("Saved threshold selection does not match this model and evaluation method")
    candidate = float(selection_metadata["candidate_threshold"])
    if not (0 <= candidate <= 1) or candidate > min(selected.values()):
        raise ValueError("Saved candidate threshold cannot recover all selected predictions")
    # This is validation provenance; V4 must not be the threshold-selection set.
    v4_hash = hashlib.sha256(v4_data.read_bytes()).hexdigest()
    if selection_metadata["data_sha256"] == v4_hash:
        raise ValueError("Thresholds were selected using the V4 data")
    tuner.filter_predictions([], selected, "recommended")  # validate threshold values

    required = [v4_per_label_out / name for name in
                ("metrics.json", "per_label_metrics.csv", "predictions.json")]
    if any(path.exists() for path in required):
        if not all(path.is_file() for path in required):
            raise FileExistsError("Partial V4 per-label result exists; existing files were preserved")
    else:
        test_data, loaded_hash = evaluator.load_data(v4_data, "recommended")
        if loaded_hash != v4_hash:
            raise ValueError("V4 data changed while being loaded")
        model = GLiNER.from_pretrained(str(saved_model), local_files_only=True)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()
        with torch.inference_mode():
            pool = evaluator.predict(model, test_data, candidate, 8, False, "recommended")
        filtered = tuner.filter_predictions(pool, selected, "recommended")
        scores = evaluator.compute_metrics(
            [row["ner"] for row in test_data],
            [[(entity["start"], entity["end"], entity["label"])
              for entity in row["predicted_entities"]] for row in filtered],
            label_schema="recommended")
        scores["thresholds"] = selected
        scores["metadata"] = {
            "model": str(saved_model), "data": str(v4_data), "data_sha256": v4_hash,
            "threshold": None, **schema_metadata("recommended"), "sample_count": len(test_data),
            "device": device, "batch_size": 8, "flat_ner": False, "multi_label": False,
            "matching": "exact inclusive token start/end + exact label within sample",
            "averaging": "micro", "zero_division": 0,
            "duplicate_predictions": "extra occurrences count as FP",
            "gliner_version": version("gliner"), "torch_version": torch.__version__,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "thresholds_file": str(threshold_file.resolve()), "thresholds_sha256": threshold_hash,
            "candidate_threshold": candidate, "filter_comparison": "score >= threshold",
            "calibration_method": selection_metadata["calibration_method"],
            "threshold_selection_metadata": selection_metadata,
            "evaluation_scope": "held-out evaluation with frozen validation-selected thresholds",
        }
        evaluator.save_results(v4_per_label_out, scores, filtered)
        del model
    per_label_v4_metrics = evaluate_or_load_v4(
        v4_per_label_out, [],
        {"threshold": None, "thresholds_sha256": threshold_hash,
         "candidate_threshold": candidate})
    if per_label_v4_metrics["thresholds"] != selected:
        raise ValueError("Saved V4 result uses different per-label thresholds")
else:
    print("Validation-selected per-label V4 evaluation skipped.")


In [ ]:
# Display saved results and write a comparison without overwriting any files.
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

v4_root = Path("/content/drive/MyDrive/Holokai_Project/final_model/outputs")
fixed = json.loads((v4_root / "v4_test/metrics.json").read_text())
calibrated = json.loads((v4_root / "v4_test_per_label/metrics.json").read_text())
for key in ("model", "data_sha256", "label_schema", "flat_ner", "multi_label", "matching", "averaging"):
    assert fixed["metadata"][key] == calibrated["metadata"][key], f"Evaluation mismatch: {key}"
assert fixed["metadata"]["threshold"] == 0.40
assert fixed["overall"]["support"] == calibrated["overall"]["support"]

fields = ["precision", "recall", "f1", "tp", "fp", "fn", "support"]
for title, result in [("Fixed threshold = 0.40", fixed),
                      ("Validation-selected per-label thresholds", calibrated)]:
    print(f"=== {title}: OVERALL ===")
    display(pd.DataFrame([result["overall"]])[fields].style.format(
        {name: "{:.2%}" for name in ("precision", "recall", "f1")}))
    print(f"=== {title}: PER-LABEL ===")
    table = pd.DataFrame.from_dict(result["per_label"], orient="index")[fields]
    table.index.name = "label"
    table.insert(0, "threshold", [result.get("thresholds", {}).get(label, 0.40) for label in table.index])
    display(table.style.format({name: "{:.2%}" for name in ("precision", "recall", "f1")}))

comparison = [{"metric": metric, "fixed_0_40": fixed["overall"][metric],
               "validation_selected_per_label": calibrated["overall"][metric],
               "change_per_label_minus_fixed": calibrated["overall"][metric] - fixed["overall"][metric]}
              for metric in ("precision", "recall", "f1", "tp", "fp", "fn")]
comparison_df = pd.DataFrame(comparison)
print("=== OVERALL COMPARISON (rates are proportions; changes are absolute) ===")
display(comparison_df)

comparison_out = v4_root / "v4_test_comparison"
comparison_out.mkdir(parents=True, exist_ok=True)
record = {"fixed_result": str(v4_root / "v4_test/metrics.json"),
          "per_label_result": str(v4_root / "v4_test_per_label/metrics.json"),
          "thresholds_sha256": calibrated["metadata"]["thresholds_sha256"], "comparison": comparison}
json_path = comparison_out / "overall_comparison.json"
if json_path.exists():
    assert json.loads(json_path.read_text()) == record, "Different saved comparison; choose a fresh comparison directory"
else:
    with json_path.open("x", encoding="utf-8") as f:
        json.dump(record, f, indent=2)
csv_path = comparison_out / "overall_comparison.csv"
if csv_path.exists():
    pd.testing.assert_frame_equal(pd.read_csv(csv_path), comparison_df, check_dtype=False)
else:
    comparison_df.to_csv(csv_path, index=False, mode="x")
print("Comparison saved (existing files preserved):", comparison_out)
